# Healthcare LLM Adaptation using Parameter-Efficient Fine-Tuning

This project demonstrates the parameter-efficient fine-tuning of a Large Language Model (LLM) for healthcare and medical-domain instruction generation using the Hugging Face ecosystem. The notebook implements a memory-efficient training pipeline using LoRA (Low-Rank Adaptation), PEFT, and 4-bit quantization techniques to adapt a pretrained LLaMA-based model on a medical conversational dataset.

The workflow covers the complete fine-tuning process, including dataset preparation, model quantization, tokenizer configuration, supervised fine-tuning with Hugging Face TRL, and inference testing. The project focuses on reducing computational requirements while maintaining strong domain adaptation performance for medical question-answering and healthcare-related conversational tasks.

## Section 1: Installing and Importing the Libraries

In [ ]:
!pip uninstall accelerate peft bitsandbytes transformers trl -y
!pip install accelerate peft bitsandbytes transformers trl==0.12.0

In [ ]:
!pip install huggingface_hub

In [ ]:
import torch
from trl import SFTTrainer
from peft import LoraConfig
from datasets import load_dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, pipeline)

## Section 2: Loading the Model

This step loads the pretrained LLaMA-based language model with memory-efficient quantization settings. The configuration enables large-scale model training while reducing GPU memory consumption.

In [ ]:
from transformers.utils import quantization_config
llama_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path="aboonaji/llama2finetune-v2",   # pretrained model
    quantization_config = BitsAndBytesConfig(load_in_4bit=True,   # 4-bit quantization to reduce model size and memory usage
                                             bnb_4bit_compute_dtype=getattr(torch, "float16"),    # data type for computation for 4-bit quantization
                                             bnb_4bit_quant_type="nf4")   # NF4 quatizination data type in the weights of the linear layer
)

# reduce memory usage and speed of computations
llama_model.config.use_cache = False    # not store output of previously computed layer in cache
llama_model.config.pretraining_tp = 1   # deactive accuracte computation of the linear layers

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/128 [00:00<?, ?it/s]

## Section 3: Loading the Tokenizer

This section initializes and configures the tokenizer used to preprocess healthcare-related text data before training. Proper tokenizer configuration ensures compatibility between the dataset and the language model.

In [ ]:
# tokenizer must use the same special tokens and padding as the model
llama_tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path="aboonaji/llama2finetune-v2",
                                                trust_remote_code=True)   # allow for custom models to be trusted

# confirgure padding (ensure pad token is same as EOS token and right padding)
llama_tokenizer.pad_token = llama_tokenizer.eos_token   # pad token that ensures all sequences will be of same length
llama_tokenizer.padding_side = "right"   # pad on the right side of the sequence

## Section 4: Setting the Training Arguments

This step defines the training configuration, including batch size, learning rate, optimization strategy, logging frequency, checkpoint handling, and other hyperparameters required for supervised fine-tuning.

In [ ]:
training_arguments = TrainingArguments(
    output_dir="./results",   # directory to save model checkpoints
    per_device_train_batch_size = 1,    # batch size for training (model processes 4 training examples each training iteration)
    max_steps = 100   # limit training to maximum 100 steps
)

## Section 5: Creating the Supervised Fine-Tuning Trainer

This section creates the Hugging Face TRL **SFTTrainer** used for supervised instruction tuning. The trainer integrates the model, dataset, tokenizer, PEFT configuration, and training arguments into a unified fine-tuning pipeline.

Supervised Fine-Tuning is a Transfer Learning technique where weights of a pre-trained model is trained on new data.

Parameter Effecient Fine-Tuning is a technique that will reduce the amount of parameter that are going to be fine-tuned instead of fully tuning all parameters of the model, which is not possible for Google Colab due to its T4 GPU size. We will be using LoRa (Low Rank Adaptation), where the **peft_config** attribute reduces the amount of parameter to be trained to the minimal amount of trainable parameters.

In [ ]:
llama_sft_trainer = SFTTrainer(
    model=llama_model,   # model to be trained
    args=training_arguments,   # training arguments
    train_dataset=load_dataset(path="aboonaji/wiki_medical_terms_llam2_format", split="train"),   # dataset to be used for training
    tokenizer=llama_tokenizer,    # tokenizer to be used
    peft_config=LoraConfig(
        task_type="CAUSAL_LM",   # specify task type as CAUSAL_LM
        r=64,   # number of trainable efficient parameters
        lora_alpha=16,    # alpha parameter for LoRa scaling
        lora_dropout=0.1   # dropout rate in the lower LoRa layers (dropout deactivates some of the weights during training)
    ),
    dataset_text_field="text",    # specify input in the training is text
    max_seq_length=256    # limit sequence lenth to reduce memory consumption
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argumen

Map:   0%|          | 0/6861 [00:00<?, ? examples/s]

## Section 6: Training the Model

This step performs the supervised fine-tuning process on the healthcare dataset. During training, the model learns domain-specific medical conversational patterns and adapts its responses to healthcare-related prompts.

In [ ]:
llama_sft_trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss


TrainOutput(global_step=100, training_loss=1.5701631164550782, metrics={'train_runtime': 97.1007, 'train_samples_per_second': 1.03, 'train_steps_per_second': 1.03, 'total_flos': 911622638641152.0, 'train_loss': 1.5701631164550782, 'epoch': 0.014575134819997084})

## Section 7: Chatting with the Model

This section tests the fine-tuned model through inference and interactive prompting. It demonstrates how the adapted healthcare LLM generates responses to medical and healthcare-related queries after training.

In [ ]:
user_prompt = "Tell me about Ascariasis"

# text generation pipeline
text_generation_pipeline = pipeline(
    task="text-generation",
    model=llama_model,
    tokenizer=llama_tokenizer,
    max_length=300    # max length of output to generate
)

model_answer = text_generation_pipeline(f"<s>[INST] {user_prompt} [/INST]")
print(model_answer[0]['generated_text'])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


<s>[INST] Tell me about Ascariasis [/INST] Ascariasis is a parasitic infection caused by the Ascaris lumbricoides tapeworm, which can infect the digestive tract of humans and other animals. nobody is immune to this disease, and it is one of the most common parasitic infections worldwide.

The Ascaris lumbricoides tapeworm is a flat, ribbon-like parasite that can grow up to 20 feet (6 meters) long in the small intestine of an infected person. The tapeworm's life cycle involves multiple hosts, including humans, pigs, dogs, and cattle. Humans become infected by ingesting eggs that are contaminated with feces, which can occur through contaminated food or water or by direct contact with feces.

Symptoms of ascariasis can include:

1. Abdominal pain
2. Diarrhea
3. Weight loss
4. Vomiting
5. Abdominal bloating
6. Constipation
7. Anemia
8. Malnutrition
9. Weakness

If left untreated, ascariasis can lead to serious complications, such as:

1. Obstruction of the
